In [0]:
# =============================================================================
# NOTEBOOK  : silver_customers
# PURPOSE   : bronze.customers → silver.customers
# LAYER     : Silver (clean, type-cast, parse, merge)
# FREQUENCY : Daily
# PATTERN   : spark.read + audit watermark (production pattern)
#             reads only new bronze rows since last successful run
#             no foreachBatch, no streaming checkpoint
#             watermark stored in audit table — fully transparent + controllable
#
# TRANSFORM:
#   - first_purchase_date  : String → DateType
#   - total_spend          : Double → Decimal(10,2)
#   - preferred_categories : raw JSON string → ArrayType(StringType)
#   - loyalty_tier         : trim and title-case for consistency
#
# MERGE & DEDUP LOGIC:
#   Dedup   : dropDuplicates on customer_id before merge
#             daily CSV could have duplicate customer records
#   Case 1  : customer_id NOT in silver → INSERT
#   Case 2  : customer_id IN silver, data changed → UPDATE
#             Changeable: age_group, gender, zip_code, loyalty_tier,
#                         total_spend, preferred_categories
#             first_purchase_date NOT updated — immutable
#   Case 3  : customer_id IN silver, no change → IGNORE
#
# REPROCESSING:
#   Full reprocess  → delete last SUCCESS entry from pipeline_audit
#   Specific window → update end_time in pipeline_audit to desired start point
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import SILVER_CUSTOMERS_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, trim, initcap, from_json
)
from pyspark.sql.types import DecimalType, ArrayType, StringType
from delta.tables import DeltaTable
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_customers"]
TARGET_TABLE = cfg["tables"]["silver_customers"]
PIPELINE     = "silver_customers"

In [0]:
# ── Read + Transform + Merge ─────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    # ── INCREMENTAL READ ──────────────────────────────────────────────────────
    # Watermark from last successful run — read only new bronze rows
    # First run → last_run_time = None → read all bronze rows
    last_run_time = get_last_successful_run_time(spark, PROJECT_ROOT, PIPELINE)
 
    if last_run_time:
        bronze_df = (
            spark.read.table(SOURCE_TABLE)
            .filter(col("ingested_at") > lit(last_run_time))
        )
    else:
        bronze_df = spark.read.table(SOURCE_TABLE)
 
    rows_read = bronze_df.count()
    print(f"[READ] {rows_read} new bronze rows since last run")
 
    if rows_read == 0:
        print(f"[SKIP] No new rows to process")
        end_audit(spark, run_id, "SUCCESS", rows_read=0, rows_written=0,
                  extra={"rows_inserted": "0", "rows_updated": "0"})
        raise SystemExit(0)
 
    # ── TRANSFORM ─────────────────────────────────────────────────────────────
    df = (
        bronze_df
 
        # 1. first_purchase_date: String → DateType
        .withColumn(
            "first_purchase_date",
            to_date(col("first_purchase_date"))
        )
 
        # 2. total_spend: Double → Decimal(10,2)
        .withColumn(
            "total_spend",
            col("total_spend").cast(DecimalType(10, 2))
        )
 
        # 3. preferred_categories: raw JSON array string → ArrayType(StringType)
        #    Source: '["Books", "Food & Beverage", "Clothing"]'
        .withColumn(
            "preferred_categories",
            from_json(
                trim(col("preferred_categories")),
                ArrayType(StringType())
            )
        )
 
        # 4. loyalty_tier: trim + title-case
        #    Standardize: "PLATINUM", "platinum", " Gold " → "Platinum", "Gold"
        .withColumn(
            "loyalty_tier",
            initcap(trim(col("loyalty_tier")))
        )
 
        # 5. Silver audit columns
        .withColumn("processed_at",    current_timestamp())
        .withColumn("pipeline_run_id", lit(run_id))
        .withColumn("source_system",   lit("customers_csv"))
 
        # 6. Enforce silver schema — drops all bronze-only columns
        #    (source_file, ingested_date, pipeline_run_id from bronze, etc.)
        .select([f.name for f in SILVER_CUSTOMERS_SCHEMA.fields])
    )
 
    # ── DEDUP ─────────────────────────────────────────────────────────────────
    # Daily CSV may contain duplicate customer records
    # dropDuplicates before merge — Delta MERGE does not dedup source df
    df = df.dropDuplicates(["customer_id"])
 
    rows_after_dedup = df.count()
    if rows_read != rows_after_dedup:
        print(f"[DEDUP] dropped {rows_read - rows_after_dedup} duplicate rows")
 
    # ── MERGE INTO SILVER ──────────────────────────────────────────────────────
    # Case 2: matched + data changed → UPDATE changeable fields only
    #         first_purchase_date excluded — immutable once set
    # Case 1: not matched → INSERT
    # Case 3: matched + no change → no rule fires → IGNORE
    (
        DeltaTable.forName(spark, TARGET_TABLE).alias("t")
        .merge(
            df.alias("s"),
            "t.customer_id = s.customer_id"
        )
        .whenMatchedUpdate(
            condition="""
                t.age_group            != s.age_group             OR
                t.gender               != s.gender                OR
                t.zip_code             != s.zip_code              OR
                t.loyalty_tier         != s.loyalty_tier          OR
                t.total_spend          != s.total_spend           OR
                t.preferred_categories != s.preferred_categories
            """,
            set={
                "age_group":            "s.age_group",
                "gender":               "s.gender",
                "zip_code":             "s.zip_code",
                "loyalty_tier":         "s.loyalty_tier",
                "total_spend":          "s.total_spend",
                "preferred_categories": "s.preferred_categories",
                "processed_at":         "current_timestamp()",
                "pipeline_run_id":      f"'{run_id}'"
                # first_purchase_date → NOT updated, immutable
                # source_system       → NOT updated, preserve original
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
 
    metrics = (
        spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
        .select("operationMetrics")
        .collect()[0][0]
    )
    rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
    rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))
 
    print(f"[DONE] inserted={rows_inserted} | updated={rows_updated}")
 
    end_audit(
        spark, PROJECT_ROOT, run_id, "SUCCESS",
        rows_read=rows_read,
        rows_written=rows_inserted + rows_updated,
        extra={
            "rows_inserted": str(rows_inserted),
            "rows_updated":  str(rows_updated)
        }
    )
 
except SystemExit:
    pass
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col
 
# 1. Audit status
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata") \
    .show(truncate=False)
 
# 2. Silver row count
print("Silver row count:", spark.read.table(TARGET_TABLE).count())
 
# 3. Nulls in key columns
print("Null customer_ids:",
    spark.read.table(TARGET_TABLE)
    .filter(col("customer_id").isNull())
    .count()
)
 
# 4. Delta history
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics") \
    .show(truncate=False)